# Practical Exam: Automating Customer Support with OpenAI API

You work as an AI Engineer at ChatSolveAI, a company that provides automated customer support solutions. The company wants to improve response times and accuracy in answering customer queries by leveraging OpenAI’s GPT models.

Your task is to build a chatbot that classifies customer queries, retrieves relevant responses, and logs interactions in a structured way. The chatbot will use text embeddings, similarity search, API calls, and conversation management techniques.


**Please note:** 

1. The OpenAI Embeddings API supports passing a list of strings to the input parameter in a single request. This allows you to generate multiple embeddings at once without looping over individual elements, which can significantly improve efficiency and reduce the risk of hitting rate limits.

2. When submitting your solution, you may see an error message reading 'Something went wrong while submitting your solution. Please try again.' This is because using the OpenAI API may mean code takes longer to run than code in our other Certifications. Please ignore this message if your code is taking a few minutes to run. However, if your code makes too many API requests, the API will time out. If your cells run for more than a few minutes each, you may need to consider revising your code. 

In [76]:
# Run this cell before running your solution

# Import necessary modules
import os
from openai import OpenAI

# Define the model to use
model = "gpt-3.5-turbo"

# Define the client
client = OpenAI()

# Task 1

ChatSolveAI has provided a knowledge base (`knowledge_base.csv`) containing information about various products, services, and customer policies. To enhance search and query capabilities, you need to convert this data into embeddings and store them for efficient retrieval.

- Load the dataset (`knowledge_base.csv`).
- Generate text embeddings using OpenAI’s embedding model (`text-embedding-3-small`). Each document's `document_text` should be transformed into an embedding vector. Do not apply any text transformations such as lowercasing, stripping or normalization before embedding.
- Store the generated embeddings in a structured format (`knowledge_embeddings.json`) with the following format available below.
- Store the embedded data and associated metadata for retrieval.  

### Format to store generated embeddings:
```json
[
    {
       "document_id": 1,
       "document_text": "Example document text.",
       "embedding_vector": [0.123, 0.456, ...],
       "metadata": "Additional document info"
    }
]
```

### Data description: 

| Column Name       | Criteria                                                |
|-------------------|---------------------------------------------------------|
| document_id       | Integer. Unique identifier for each document. No missing values. |
| document_text     | String. Text content of the knowledge base. Preprocessed and embedded. |
| embedding_vector  | List. Embedding representation of the `document_text`. |
| metadata          | String. Metadata for additional information. |


In [77]:
# Write your answer to Task 1 here
# Import necessary libraries
import pandas as pd
import json
import time
from typing import List, Dict, Any

# Load the dataset
knowledge_base = pd.read_csv("knowledge_base.csv")
print(f"Loaded {len(knowledge_base)} documents from knowledge_base.csv")

# Function to generate embeddings with error handling and rate limiting
def generate_embeddings_batch(texts: List[str], model: str ="text-embedding-3-small", batch_size: int = 100) -> List[List[float]]:
    """
    Generate embeddings for a list of texts using OpenAI API with batch processing
    """
    batch_embeddings = []
    for i in range(0, len(texts), batch_size):
        batch = texts[i:i + batch_size]

        try: 
            # Generate embeddings for the batch
            response = client.embeddings.create(
                input=batch,
                model=model
            )
            
            # Extract embeddings from response
            embeddings = [response_embedding.embedding for response_embedding in response.data]
            batch_embeddings.extend(embeddings)

            print(f"Generated embeddings for batch {i//batch_size + 1}/{(len(texts)-1)//batch_size + 1}")
            
            # Rate limiting - small delay between batches
            time.sleep(0.1)
            
        except Exception as e:
            print(f"Error generating embeddings for batch {i//batch_size + 1}: {e}")
            # Retry with exponential backoff
            time.sleep(2)
            try:
                response = client.embeddings.create(
                    input=batch,
                    model=model
                )
                embeddings = [response_embedding.embedding for response_embedding in response.data]
                batch_embeddings.extend(embeddings)
                print(f"Retry successful for batch {i//batch_size + 1}")
            except Exception as retry_error:
                print(f"Retry failed for batch {i//batch_size + 1}: {retry_error}")
                
                # Add empty embeddings as placeholder
                batch_embeddings.extend([[] for _ in batch])
                
    return batch_embeddings

# Generate text embeddings
document_texts = knowledge_base["document_text"].tolist()
print("Generating text embeddings...")
document_embeddings = generate_embeddings_batch(document_texts)

# Create structured format for storing embeddings
knowledge_embeddings = []
for i, row in knowledge_base.iterrows():
    document_id = int(row["document_id"])
    document_text = row["document_text"]
    document_metadata = row["metadata"]
    knowledge_embedding = {
       "document_id": document_id,
       "document_text": document_text,
       "embedding_vector": document_embeddings[i],
       "metadata": document_metadata
    }
    knowledge_embeddings.append(knowledge_embedding)

# Save embeddings to JSON file
with open("knowledge_embeddings.json", "w") as knowledge_embeddings_file:
    json.dump(knowledge_embeddings, knowledge_embeddings_file, indent=2)

print(f"Successfully generated and saved {len(knowledge_embeddings)} embeddings to knowledge_embeddings.json")
print(f"Sample embedding vector length: {len(document_embeddings[0]) if document_embeddings[0] else 0}")
pd.read_json("knowledge_embeddings.json").head(5)

Loaded 501 documents from knowledge_base.csv
Generating text embeddings...
Generated embeddings for batch 1/6
Generated embeddings for batch 2/6
Generated embeddings for batch 3/6
Generated embeddings for batch 4/6
Generated embeddings for batch 5/6
Generated embeddings for batch 6/6
Successfully generated and saved 501 embeddings to knowledge_embeddings.json
Sample embedding vector length: 1536


,document_id,document_text,embedding_vector,metadata
0,1,This document provides details about payment m...,"[0.005001068115234, 0.016098022460937, 0.02349...",Requires proof of purchase
1,2,This document provides details about troublesh...,"[-0.011009216308593, 0.05755615234375, 0.06179...",Subject to availability
2,3,This document provides details about refund po...,"[0.000640869140625, 0.05618286132812501, 0.016...",Limited to US customers
3,4,This document provides details about product w...,"[0.008140563964843, 0.05239868164062501, 0.031...",Customer service contact required
4,5,This document provides details about product w...,"[0.013008117675781, 0.0587158203125, 0.0186767...",Requires account verification


# Task 2

ChatSolveAI receives customer queries that need to be classified and matched with appropriate responses. Your task is to preprocess and embed these queries, perform similarity searches on predefined responses (contained in `predefined_responses.json`), and retrieve the most relevant responses.

- Load the dataset (`processed_queries.csv`).
- Retrieve responses by using cosine similarity to perform a similarity search against predefined responses in `predefined_responses.json`.
- Structure API requests properly and implement error handling, including retry mechanisms to handle rate limits.
- Format model responses as JSON to maintain consistency in output.
- Compute confidence scores for retrieved responses, scaled to 0-1.
- Store the structured responses in a JSON file (`query_responses.json`), suitable for integration with other applications. Your JSON file should be structured as follows:

| Column Name       | Criteria                                                   |
|-------------------|------------------------------------------------------------|
| query_id         | Integer. Unique identifier for each query. No missing values. |
| query_text       | String. Preprocessed query text. |
| top_responses    | List. Top 3 most relevant response strings retrieved. |
| confidence_scores | List. Model-based confidence score for the top 3 responses. |

In [78]:
# Write your answer to Task 2 here
import pandas as pd
import json
import numpy as np
import time
from sklearn.metrics.pairwise import cosine_similarity
from typing import List, Dict, Any, Tuple

# Load the processed queries dataset
processed_queries = pd.read_csv('processed_queries.csv')
print(f"Loaded {len(processed_queries)} queries from processed_queries.csv")

# Load predefined responses
with open('predefined_responses.json', 'r') as predefined_file:
    predefined_responses = json.load(predefined_file)

print(f"Loaded {len(predefined_responses)} predefined responses")

# Function to generate embeddings with retry mechanism
def generate_embeddings_with_retry(texts: List[str], model: str = "text-embedding-3-small", max_retries: int = 3) -> List[List[float]]:
    """
    Generate embeddings with error handling and retry mechanism
    """
    for attempt in range(max_retries):
        try:
            response = client.embeddings.create(
                input=texts,
                model=model
            )
            return [embedding.embedding for embedding in response.data]
        except Exception as e:
            print(f"Attempt {attempt + 1} failed: {e}")
            if attempt < max_retries - 1:
                wait_time = 2 ** attempt  # Exponential backoff
                print(f"Retrying in {wait_time} seconds...")
                time.sleep(wait_time)
            else:
                print("Max retries reached. Returning empty embeddings.")
                return [[] for _ in texts]
    return []

# Generate embeddings for queries in batches
def process_queries_in_batches(queries: List[str], batch_size: int = 50) -> List[List[float]]:
    """
    Process queries in batches to handle rate limits
    """
    all_embeddings = []
    
    for i in range(0, len(queries), batch_size):
        batch = queries[i:i + batch_size]
        print(f"Processing batch {i//batch_size + 1}/{(len(queries)-1)//batch_size + 1}")
        
        batch_embeddings = generate_embeddings_with_retry(batch)
        all_embeddings.extend(batch_embeddings)
        
        # Rate limiting
        time.sleep(0.2)
    
    return all_embeddings

# Generate embeddings for queries
query_texts = queries_df['query_text'].tolist()
print("Generating embeddings for queries...")
query_embeddings = process_queries_in_batches(query_texts)

# Generate embeddings for predefined responses
response_texts = list(predefined_responses.values())
response_keys = list(predefined_responses.keys())
print("Generating embeddings for predefined responses...")
response_embeddings = generate_embeddings_with_retry(response_texts)

# Function to find top similar responses using cosine similarity
def find_top_responses(query_embedding: List[float], response_embeddings: List[List[float]], 
                      response_texts: List[str], response_keys: List[str], top_k: int = 3) -> Tuple[List[str], List[float]]:
    """
    Find top k most similar responses using cosine similarity
    """
    if not query_embedding or not any(response_embeddings):
        # Return default responses if embeddings are empty
        return response_texts[:top_k], [0.5] * min(top_k, len(response_texts))
    
    # Calculate cosine similarities
    query_embedding = np.array(query_embedding).reshape(1, -1)
    response_embeddings_array = np.array([emb for emb in response_embeddings if emb])
    
    if response_embeddings_array.size == 0:
        return response_texts[:top_k], [0.5] * min(top_k, len(response_texts))
    
    similarities = cosine_similarity(query_embedding, response_embeddings_array)[0]
    
    # Get top k indices
    top_indices = np.argsort(similarities)[::-1][:top_k]
    
    # Get corresponding responses and confidence scores
    top_responses = [response_texts[i] for i in top_indices]
    confidence_scores = [(similarities[i] + 1) / 2 for i in top_indices]  # Scale to 0-1
    
    return top_responses, confidence_scores

# Process all queries and find top responses
query_responses = []

print("Finding top responses for each query...")
for idx, row in queries_df.iterrows():
    query_id = int(row['query_id'])
    query_text = row['query_text']
    query_embedding = query_embeddings[idx] if idx < len(query_embeddings) else []
    
    # Find top 3 responses
    top_responses, confidence_scores = find_top_responses(
        query_embedding, response_embeddings, response_texts, response_keys, top_k=3
    )
    
    # Create response entry
    response_entry = {
        "query_id": query_id,
        "query_text": query_text,
        "top_responses": top_responses,
        "confidence_scores": confidence_scores
    }
    
    query_responses.append(response_entry)
    
    if (idx + 1) % 50 == 0:
        print(f"Processed {idx + 1}/{len(queries_df)} queries")

# Save structured responses to JSON file
with open('query_responses.json', 'w') as f:
    json.dump(query_responses, f, indent=2)

print(f"Successfully processed {len(query_responses)} queries and saved to query_responses.json")
print(f"Sample response: {query_responses[0] if query_responses else 'No responses generated'}")
pd.read_json("query_responses.json").head(5)

Loaded 501 queries from processed_queries.csv
Loaded 19 predefined responses
Generating embeddings for queries...
Processing batch 1/11
Processing batch 2/11
Processing batch 3/11
Processing batch 4/11
Processing batch 5/11
Processing batch 6/11
Processing batch 7/11
Processing batch 8/11
Processing batch 9/11
Processing batch 10/11
Processing batch 11/11
Generating embeddings for predefined responses...
Finding top responses for each query...
Processed 50/501 queries
Processed 100/501 queries
Processed 150/501 queries
Processed 200/501 queries
Processed 250/501 queries
Processed 300/501 queries
Processed 350/501 queries
Processed 400/501 queries
Processed 450/501 queries
Processed 500/501 queries
Successfully processed 501 queries and saved to query_responses.json
Sample response: {'query_id': 1, 'query_text': 'How can I contact customer support?', 'top_responses': ['You can contact our customer support via email or live chat on our website.', 'If you received a damaged product, pleas

,query_id,query_text,top_responses,confidence_scores
0,1,How can I contact customer support?,[You can contact our customer support via emai...,"[0.8336312202274551, 0.6863651994530351, 0.665..."
1,2,can i purchase a gift card,"[Yes, we offer gift cards in various denominat...","[0.8107559919331151, 0.7114608670326971, 0.682..."
2,3,"My order hasn't arrived yet, what should I do?",[Track your order by logging into your account...,"[0.7529436644103601, 0.6713502787818351, 0.666..."
3,4,How do I reset my password?,[You can reset your password by clicking on 'F...,"[0.8283802141065011, 0.6852298424816551, 0.671..."
4,5,how do i track my order,[Track your order by logging into your account...,"[0.8734820483124031, 0.690186375173064, 0.6735..."


# Task 3

To provide seamless customer service, ChatSolveAI wants to develop a chatbot that can respond to customer queries efficiently by searching for relevant responses and generating new ones when necessary.

- Develop a chatbot that:
    - Accepts customer queries via text input.
    - Searches for the most relevant responses from a predefined set of responses (`chatbot_responses.json`).
    - Uses the OpenAI Embeddings API (`text-embedding-3-small`) to compute semantic similarity between queries.
    - If no relevant response is found from the predefined set, generates a new response using GPT-3.5-turbo.
- Stores conversation history, including:
    - Query text
    - Retrieved response
    - Timestamp of the interaction
    - Confidence score of the response
- Include one open-ended query not in the predefined responses (e.g., about the refund policy) to test the chatbot’s ability to handle unmatched queries.
- Include one paraphrased query about support hours (e.g., “When can I talk to someone from support?”) to test semantic similarity matching.
- Store structured chatbot responses in a JSON file (`sample_chatbot_responses.json`). Make sure they follow this format:
```json
[
    {
        "query_text": "How do I reset my password?",
        "retrieved_response": "You can reset your password by clicking 'Forgot Password' on the login page.",
        "timestamp": "2025-04-02T14:30:00Z",
        "confidence_score": 0.92
    },
    {
        "query_text": "What are your business hours?",
        "retrieved_response": "Our support team is available from 9 AM to 5 PM, Monday to Friday.",
        "timestamp": "2025-04-02T14:35:00Z",
        "confidence_score": 0.87
    }
]
```

In [79]:
# Write your answer to Task 3 here
# Import necessary libraries
import pandas as pd
import json
import numpy as np
import time
from datetime import datetime
from sklearn.metrics.pairwise import cosine_similarity
from typing import List, Dict, Any, Tuple

# Create customer support chatbot class
class CustomerSupportChatbot:
    def __init__(self, predefined_responses_file: str, similarity_threshold: float = 0.7):
        """
        Initialize the chatbot with predefined responses and similarity threshold
        """
        self.similarity_threshold = similarity_threshold
        self.conversation_history = []
        
        # Load predefined responses from chatbot_responses.json
        with open(predefined_responses_file, 'r') as f:
            self.predefined_responses = json.load(f)
        
        # Extract query texts and responses for similarity matching
        self.predefined_queries = [item['query_text'] for item in self.predefined_responses]
        self.predefined_answers = [item['retrieved_response'] for item in self.predefined_responses]
        
        # Generate embeddings for predefined queries
        self.predefined_embeddings = self._generate_embeddings(self.predefined_queries)
        
        print(f"Chatbot initialized with {len(self.predefined_responses)} predefined responses")
    
    def _generate_embeddings(self, texts: List[str]) -> List[List[float]]:
        """
        Generate embeddings for a list of texts using OpenAI API
        """
        try:
            response = client.embeddings.create(
                input=texts,
                model="text-embedding-3-small"
            )
            return [embedding.embedding for embedding in response.data]
        except Exception as e:
            print(f"Error generating embeddings: {e}")
            return [[] for _ in texts]
    
    def _find_best_match(self, query_embedding: List[float]) -> Tuple[str, float]:
        """
        Find the best matching response using cosine similarity
        """
        if not query_embedding or not any(self.predefined_embeddings):
            return None, 0.0
        
        # Calculate cosine similarities
        query_embedding = np.array(query_embedding).reshape(1, -1)
        predefined_embeddings_array = np.array([emb for emb in self.predefined_embeddings if emb])
        
        if predefined_embeddings_array.size == 0:
            return None, 0.0
        
        similarities = cosine_similarity(query_embedding, predefined_embeddings_array)[0]
        
        # Find the best match
        best_idx = np.argmax(similarities)
        best_similarity = similarities[best_idx]
        
        # Convert cosine similarity to confidence score (0-1 range)
        confidence_score = (best_similarity + 1) / 2
        
        if confidence_score >= self.similarity_threshold:
            return self.predefined_answers[best_idx], confidence_score
        else:
            return None, confidence_score
    
    def _generate_new_response(self, query: str) -> str:
        """
        Generate a new response using GPT-3.5-turbo when no relevant predefined response is found
        """
        try:
            response = client.chat.completions.create(
                model="gpt-3.5-turbo",
                messages=[
                    {
                        "role": "system",
                        "content": "You are a helpful customer support assistant. Provide clear, concise, and helpful responses to customer queries. If you don't have specific information, provide general guidance and suggest contacting support for detailed assistance."
                    },
                    {
                        "role": "user",
                        "content": query
                    }
                ],
                max_tokens=150,
                temperature=0.7
            )
            return response.choices[0].message.content.strip()
        except Exception as e:
            print(f"Error generating new response: {e}")
            return "I apologize, but I'm having trouble processing your request right now. Please contact our support team for assistance."
    
    def process_query(self, query: str) -> Dict[str, Any]:
        """
        Process a customer query and return the response with metadata
        """
        timestamp = datetime.utcnow().isoformat() + 'Z'
        
        # Generate embedding for the query
        query_embeddings = self._generate_embeddings([query])
        query_embedding = query_embeddings[0] if query_embeddings else []
        
        # Try to find a matching predefined response
        matched_response, confidence_score = self._find_best_match(query_embedding)
        
        if matched_response:
            response_text = matched_response
            print(f"Found matching response with confidence: {confidence_score:.3f}")
        else:
            # Generate new response using GPT-3.5-turbo
            response_text = self._generate_new_response(query)
            confidence_score = 0.5  # Default confidence for generated responses
            print(f"Generated new response (no match found, confidence was: {confidence_score:.3f})")
        
        # Create response entry
        response_entry = {
            "query_text": query,
            "retrieved_response": response_text,
            "timestamp": timestamp,
            "confidence_score": round(confidence_score, 3)
        }
        
        # Add to conversation history
        self.conversation_history.append(response_entry)
        return response_entry
    
    def save_conversation_history(self, filename: str):
        """
        Save conversation history to a JSON file
        """
        with open(filename, 'w') as conversation_history_file:
            json.dump(self.conversation_history, conversation_history_file, indent=2)
        print(f"Conversation history saved to {filename}")

# Initialize the chatbot
chatbot = CustomerSupportChatbot('chatbot_responses.json', similarity_threshold=0.7)

# Test queries as specified in the task
test_queries = [
    "How do I reset my password?",  # Should match predefined response
    "When can I talk to someone from support?",  # Paraphrased query about support hours
    "What is your detailed refund policy for digital products?",  # Open-ended query not in predefined responses
    "How can I contact customer support?",  # Should match predefined response
    "My order hasn't arrived yet, what should I do?",  # Should match predefined response
]

print("\n=== Processing Test Queries ===")
for i, query in enumerate(test_queries, 1):
    print(f"\nQuery {i}: {query}")
    response = chatbot.process_query(query)
    print(f"Response: {response['retrieved_response']}")
    print(f"Confidence: {response['confidence_score']}")
    print(f"Timestamp: {response['timestamp']}")

# Save the conversation history
chatbot.save_conversation_history('sample_chatbot_responses.json')

print(f"\n=== Chatbot Testing Complete ===")
print(f"Total queries processed: {len(chatbot.conversation_history)}")
print(f"Conversation history saved to: sample_chatbot_responses.json")

# Display the final conversation history
print("\n=== Final Conversation History ===")
for entry in chatbot.conversation_history:
    print(f"Query: {entry['query_text']}")
    print(f"Response: {entry['retrieved_response']}")
    print(f"Confidence: {entry['confidence_score']}")
    print(f"Timestamp: {entry['timestamp']}")
    print("-" * 50)

Chatbot initialized with 19 predefined responses

=== Processing Test Queries ===

Query 1: How do I reset my password?
Found matching response with confidence: 0.913
Response: You can reset your password by clicking 'Forgot Password' on the login page.
Confidence: 0.913
Timestamp: 2026-07-18T01:51:24.772632Z

Query 2: When can I talk to someone from support?
Found matching response with confidence: 0.841
Response: Our support team is available from 9 AM to 5 PM, Monday to Friday.
Confidence: 0.841
Timestamp: 2026-07-18T01:51:25.029699Z

Query 3: What is your detailed refund policy for digital products?
Found matching response with confidence: 0.766
Response: Yes, refunds are available within 30 days of purchase if you meet our refund policy criteria.
Confidence: 0.766
Timestamp: 2026-07-18T01:51:25.241137Z

Query 4: How can I contact customer support?
Found matching response with confidence: 0.805
Response: You can contact technical support via our live chat or email at support@exampl